# 8.2 商品期货动量策略（CTA）

## 学习目标
- 了解 CTA（Commodity Trading Advisor）的策略框架
- 实现时间序列动量（TSMOM）和截面动量策略
- 分析商品期货动量因子的 IC 和风险特征


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
np.random.seed(42)
print('Libraries loaded')


## 1. 什么是 CTA 策略？

CTA 是专注于期货市场的量化策略，核心是**趋势跟踪**：
- 识别资产价格的中长期趋势
- 在趋势确立后顺势持仓（多/空均可）
- 在趋势反转时止损或反向

**CTA 的最大优势**：与股票市场相关性低，在股市危机时往往有正收益（「危机 Alpha」）。


In [ ]:
# 模拟 10 个商品期货品种的日收益率
np.random.seed(42)
n_assets = 10
n_days = 252 * 5
asset_names = ['原油','黄金','铜','玉米','大豆','棉花','橡胶','螺纹钢','铁矿石','豆粕']

# 各品种有不同的趋势特征
returns_df = pd.DataFrame()
for i, name in enumerate(asset_names):
    trend_strength = np.random.uniform(0.0001, 0.0005)
    trend = trend_strength * np.sin(np.linspace(0, 6*np.pi, n_days) + i)
    noise = np.random.normal(0, 0.015, n_days)
    returns_df[name] = trend + noise

print(f'模拟商品期货收益矩阵: {returns_df.shape}')


## 2. 时间序列动量（TSMOM）策略


In [ ]:
lookback = 63  # 3个月动量窗口

# 计算各品种的信号方向（+1做多/-1做空）
signals = pd.DataFrame(index=returns_df.index, columns=returns_df.columns)
for col in returns_df.columns:
    roll_ret = returns_df[col].rolling(lookback).sum()
    signals[col] = np.sign(roll_ret)  # +1正动量做多, -1负动量做空

# 等波动率加权的策略收益
vol_target = 0.01  # 每日目标波动率 1%
roll_vol = returns_df.rolling(21).std()
weights = vol_target / (roll_vol + 1e-10)
weights = weights.clip(upper=5)  # 最大杠杆限制

strat_returns = (signals.shift(1) * weights.shift(1) * returns_df).mean(axis=1)
cum_strat  = (1 + strat_returns).cumprod()

plt.figure(figsize=(12, 5))
plt.plot(cum_strat, 'b-', lw=1.5, label='TSMOM CTA 策略')
plt.axhline(1, color='gray', linestyle='--', alpha=0.5)
plt.title('时间序列动量（TSMOM）CTA 策略净值')
plt.ylabel('累积净值'); plt.legend(); plt.grid(alpha=0.3); plt.show()
ann_ret = strat_returns.mean() * 252
ann_vol = strat_returns.std() * np.sqrt(252)
print(f'年化收益: {ann_ret:.2%}   年化波动率: {ann_vol:.2%}   Sharpe: {ann_ret/ann_vol:.3f}')


## 3. 截面动量：品种间排名策略


In [ ]:
# 按过去 1 个月动量排名，做多前 3 名，做空后 3 名
cross_sec_ret = []
for t in range(lookback, n_days):
    mom = returns_df.iloc[t-lookback:t].sum()
    rank = mom.rank()
    w = pd.Series(0.0, index=returns_df.columns)
    w[rank >= rank.quantile(0.7)] = 1/3   # 多头
    w[rank <= rank.quantile(0.3)] = -1/3  # 空头
    r = (w * returns_df.iloc[t]).sum()
    cross_sec_ret.append(r)

cs_series = pd.Series(cross_sec_ret)
cum_cs = (1 + cs_series).cumprod()
plt.figure(figsize=(10, 4))
plt.plot(cum_cs, 'darkorange', lw=1.5, label='截面动量策略')
plt.title('截面动量：月频排名多空组合')
plt.ylabel('累积净值'); plt.legend(); plt.grid(alpha=0.3); plt.show()


## 🎯 练习

1. 将 TSMOM 的回望窗口从 3 个月扩展到 12 个月，观察哪个窗口的 Sharpe 最高。
2. 对比 CTA 策略在 2008 年金融危机（模拟）期间的表现与股票组合的表现，验证「危机 Alpha」。
3. 研究 AQR 的 TSMOM 论文（Moskowitz等，2012），复现其在 58 个期货品种上的回测结论。

---
**下一节** → `03_term_structure_strategy.ipynb`